# I. Business Understanding

Our company, SyriaTel, is a telecommunications company based and operating in Syria, and is looking to better understand the patterns shown by customers who are likely to switch to another telecom company, referred to as "customer churn." While it will be impossible to predict with 100% confidence who will or won't churn, not least of which due to a variety of confounding factors relating to more than 13 years of civil war and the recent collapse of the ruling regime, we can search for predictable patterns within customer data that would highlight potential warning signs that a SyriaTel customer may be on the verge of churning. 

The essential questions for us to address are:
1. What factors indicate that a customer may churn?
2. What can SyriaTel do to minimize the likelihood of churning?

# II. Data Understanding

To answer these questions, we will treat the problem as one of binary classification and construct a predictive model. In order to maximize analytical efficacy, we will utilize several different machine learning models for classification and determine which model is more useful by comparing scores assessing the models' performance. 

### False Negatives and False Positives
- A positive prediction entails that the customer **is** likely to churn.
- A negative prediction indicates a customer **is not** likely to churn.

While we must be wary of both False Positives (FPs) and False Negatives (FNs) in our predictive models, since our purpose is to minimize customer churn we must prioritize minimizing rates of False Negatives, as falsely predicting a customer is unlikely to churn means that SyriaTel will fail to act to prevent this churn, whereas a False Positive will cost the company less in the long run as the customer will not churn.

### Model Scoring
For this reason, we will prioritize comparing the following in evaluating the various models:
1. `Recall` score, also known as "sensitivity", which essentially evaluates how successful our models were in actually predicting Positives. In other words, **a high Recall score means fewer False Negatives.**
2. `AUC-ROC` [Area Under the Curve - Receiver Operator Characteristic) score, which essentially provides a scaled value that determines how effective our models were in distinguishing between Positive and Negative outcomes.

### Dataset: Features & Target
The dataset we will be working with, which can be found [here](https://www.kaggle.com/datasets/becksddf/churn-in-telecoms-dataset/discussion?sort=hotness), is pulled from Kaggle.

The dataset contains records for 3,333 customers, and 21 columns with a variety of information about the customer.

Our **Target** column will be `churn`, which contains boolean values of whether the customer churned or not. The dataset indicates that about 14% of customers did churn, while 86% did not churn.

This leaves 20 columns to be considered as **Features**. While the dataset includes a number of Categorial Variables (object datatypes), the majority of columns are Discrete Numeric (integer or float datatypes). Luckily, the dataset contains no null values.

## II.a. Data Preparation

First, we need to import the libraries required for this project, and inspect the dataset.

In [92]:
# importing necessary libraries

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

from sklearn.model_selection import train_test_split, cross_validate, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve, get_scorer, classification_report

from imblearn.over_sampling import SMOTE


In [6]:
# loading in the dataset
df = pd.read_csv("data/SyriaTel_Data.csv")
df.head()

,state,account length,area code,phone number,international plan,voice mail plan,number vmail messages,total day minutes,total day calls,total day charge,...,total eve calls,total eve charge,total night minutes,total night calls,total night charge,total intl minutes,total intl calls,total intl charge,customer service calls,churn
0,KS,128,415,382-4657,no,yes,25,265.1,110,45.07,...,99,16.78,244.7,91,11.01,10.0,3,2.70,1,False
1,OH,107,415,371-7191,no,yes,26,161.6,123,27.47,...,103,16.62,254.4,103,11.45,13.7,3,3.70,1,False
2,NJ,137,415,358-1921,no,no,0,243.4,114,41.38,...,110,10.30,162.6,104,7.32,12.2,5,3.29,0,False
3,OH,84,408,375-9999,yes,no,0,299.4,71,50.90,...,88,5.26,196.9,89,8.86,6.6,7,1.78,2,False
4,OK,75,415,330-6626,yes,no,0,166.7,113,28.34,...,122,12.61,186.9,121,8.41,10.1,3,2.73,3,False


### Metadata for the Columns, pulled from Kaggle

`state`: The state of the customer.\
`account length`: The length of the account in days or months.\
`area code`: The area code of the customer's phone number.\
`phone number`: The phone number of the customer.\
`international plan`: Whether the customer has an international plan or not.\
`voice mail plan`: Whether the customer has a voicemail plan or not.\
`number vmail messages`: The number of voicemail messages the customer has.\
`total day minutes`: Total minutes of day calls.\
`total day calls`: Total number of day calls.\
`total day charge`: Total charge for the day calls.\
`total eve minutes`: Total minutes of evening calls.\
`total eve calls`: Total number of evening calls.\
`total eve charge`: Total charge for the evening calls.\
`total night minutes`: Total minutes of night calls.\
`total night calls`: Total number of night calls.\
`total night charge`: Total charge for the night calls.\
`total intl minutes`: Total minutes of international calls.\
`total intl calls`: Total number of international calls.\
`total intl charge`: Total charge for the international calls.\
`customer service calls`: Number of times the customer called customer service.\
`churn`: Whether the customer churned or not (True/False).

In [7]:
# Inspecting the number of customers (rows) and the number of features (columns)
print(f'Rows, Columns: {df.shape}')
print()
print(f'Columns: {df.columns}')
print()
print(df.info())

Rows, Columns: (3333, 21)

Columns: Index(['state', 'account length', 'area code', 'phone number',
       'international plan', 'voice mail plan', 'number vmail messages',
       'total day minutes', 'total day calls', 'total day charge',
       'total eve minutes', 'total eve calls', 'total eve charge',
       'total night minutes', 'total night calls', 'total night charge',
       'total intl minutes', 'total intl calls', 'total intl charge',
       'customer service calls', 'churn'],
      dtype='object')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3333 entries, 0 to 3332
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   state                   3333 non-null   object 
 1   account length          3333 non-null   int64  
 2   area code               3333 non-null   int64  
 3   phone number            3333 non-null   object 
 4   international plan      3333 non-null   object 
 5   vo

In [13]:
# Listing the Categorical Variables
categorical_columns = df.select_dtypes(include=['object']).columns
print(f"Categorical Columns: {categorical_columns}")

Categorical Columns: Index(['state', 'phone number', 'international plan', 'voice mail plan'], dtype='object')


While `international plan` or `voice mail plan` certainly would be of us to our analysis as they indicate which services the customers subscribe to, the `state` and `phone number` of each customer are probably far less useful to our analysis.

#### Are there any null values we need to be concerned about?

In [18]:
df.isna().sum()

state                     0
account length            0
area code                 0
phone number              0
international plan        0
voice mail plan           0
number vmail messages     0
total day minutes         0
total day calls           0
total day charge          0
total eve minutes         0
total eve calls           0
total eve charge          0
total night minutes       0
total night calls         0
total night charge        0
total intl minutes        0
total intl calls          0
total intl charge         0
customer service calls    0
churn                     0
dtype: int64

#### Is there an imbalance in the Target variable?

In [11]:
# Looking at how many records indicate customer churning vs. no churning
print(df.churn.value_counts())
print()
print(df.churn.value_counts(normalize=True))

churn
False    2850
True      483
Name: count, dtype: int64

churn
False    0.855086
True     0.144914
Name: proportion, dtype: float64


It seems there is heavy imbalance in the `churn` column. If needed, we can try to counter this imbalance using `SMOTE` (synthetic minority oversampling) which would deal with class imbalances by oversampling the minority class.

#### Adjusting the Dataset: Dropping Columns and Changing Datatypes

In [27]:
# Copying the dataframe
df_clean = df.copy()

# Changing "yes/no" inputs with "1/0" numeric inputs
df_clean['voice mail plan'] = df_clean['voice mail plan'].replace({'yes': 1, 'no': 0})
df_clean['international plan'] = df_clean['international plan'].replace({'yes': 1, 'no': 0})

# Dropping unnecessary columns
df_clean = df_clean.drop(['state', 'phone number'], axis=1)

df_clean.head()

,account length,area code,international plan,voice mail plan,number vmail messages,total day minutes,total day calls,total day charge,total eve minutes,total eve calls,total eve charge,total night minutes,total night calls,total night charge,total intl minutes,total intl calls,total intl charge,customer service calls,churn
0,128,415,0,1,25,265.1,110,45.07,197.4,99,16.78,244.7,91,11.01,10.0,3,2.70,1,False
1,107,415,0,1,26,161.6,123,27.47,195.5,103,16.62,254.4,103,11.45,13.7,3,3.70,1,False
2,137,415,0,0,0,243.4,114,41.38,121.2,110,10.30,162.6,104,7.32,12.2,5,3.29,0,False
3,84,408,1,0,0,299.4,71,50.90,61.9,88,5.26,196.9,89,8.86,6.6,7,1.78,2,False
4,75,415,1,0,0,166.7,113,28.34,148.3,122,12.61,186.9,121,8.41,10.1,3,2.73,3,False


#### Assigning, Splitting, and Scaling the Variables

In [28]:
X = df_clean.copy().drop('churn', axis=1)
y = df_clean['churn'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)



In [35]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Turning the scaled X_train and X_test arrays back into Dataframes
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns, index=X_test.index)

In [34]:
X_test_scaled

,account length,area code,international plan,voice mail plan,number vmail messages,total day minutes,total day calls,total day charge,total eve minutes,total eve calls,total eve charge,total night minutes,total night calls,total night charge,total intl minutes,total intl calls,total intl charge,customer service calls
438,0.315791,1.749923,-0.327448,-0.611418,-0.584700,-0.462675,-0.372733,-0.462730,2.562862,0.301688,2.562574,-0.220713,1.183057,-0.222038,1.159229,-0.595235,1.165138,-0.427903
2674,-0.847941,-0.512381,-0.327448,-0.611418,-0.584700,-1.311946,0.829797,-1.311676,0.326524,1.198556,0.326702,-0.240382,2.105062,-0.239521,0.908959,0.638857,0.913550,-1.180421
1345,-0.063687,-0.512381,-0.327448,-0.611418,-0.584700,-3.330584,-5.032539,-3.330643,-0.815352,1.497512,-0.814476,-0.659343,-0.609730,-0.659130,-1.236214,-1.417963,-1.231568,1.829653
1957,1.175941,-0.679077,-0.327448,-0.611418,-0.584700,0.606778,-1.074209,0.607160,0.063774,-0.445702,0.064068,-0.873741,0.670832,-0.873306,-0.020616,-1.006599,-0.026594,-0.427903
2148,-0.114284,-0.679077,-0.327448,-0.611418,-0.584700,-0.666204,0.078216,-0.666259,0.470740,-1.342571,0.470802,0.532630,-0.456063,0.534133,-0.092122,1.050221,-0.092802,-0.427903
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3257,1.783105,-0.512381,-0.327448,-0.611418,-0.584700,-0.786471,0.479059,-0.785982,-0.054760,0.451166,-0.054466,1.811150,1.592837,1.810444,1.087723,0.227493,1.085690,0.324616
1586,-0.291373,-0.512381,-0.327448,-0.611418,-0.584700,-1.807817,-1.174420,-1.807982,-0.665209,-0.993789,-0.665728,-0.116464,-1.531735,-0.117135,-1.093203,-0.183871,-1.099154,-0.427903
3068,-0.569657,-0.512381,-0.327448,1.635543,0.952907,-0.359060,-0.773577,-0.359332,0.439131,-1.043615,0.438263,-1.507100,-0.404840,-1.507090,-0.270886,-1.417963,-0.264941,0.324616
2484,1.024150,-0.512381,-0.327448,1.635543,2.270856,-1.167625,1.330852,-1.168008,1.494082,-0.595181,1.493446,1.756076,1.285502,1.757993,0.730194,-1.006599,0.728170,-1.180421


# III. Exploratory Data Analysis

## III.a. Modeling

Since we want to compare the performance of a few models, we'll need to create a function capable of determining the ROC-AUC and Recall scores and produces a legible dataframe for us to easily read. 

This function will calculate those two metrics through the training data, as well as through cross-validation of five subsets. By comparing the scores of these two sets, we can further assess degrees of overfitting or underfitting. 

We will also want to use this function on the test set eventually, in which case there will be no need for cross-validation and the `cv` parameter can be set to False.

In [38]:
def model_metric_results(estimator, X, y, metrics=['roc_auc', 'recall'], cv=True, k=5):
    train_scores_list = []
    cv_scores_list = []

    for m in metrics:
        scorer = get_scorer(m)
        train_score = scorer(estimator, X, y)
        train_scores_list.append(train_score)

    if cv:
        cv_results = cross_validate(estimator, X, y, cv=k, scoring=metrics)
        for m in metrics:
            cv_mean = np.mean(cv_results['test_' + m])
            cv_scores_list.append(cv_mean)
        df = pd.DataFrame(zip(train_scores_list, cv_scores_list), columns=['Training_Score', 'Mean_CV_Score'], index=metrics)
        return df
    else:
        df = pd.DataFrame(data=train_scores_list, columns=['Score'], index=metrics)
        return df

### Logistic Regression

In [39]:
logreg = LogisticRegression(random_state=42)

logreg.fit(X_train_scaled, y_train)

LogisticRegression(random_state=42)

In [40]:
logreg_results = model_metric_results(logreg, X_train_scaled, y_train)
logreg_results

,Training_Score,Mean_CV_Score
roc_auc,0.818385,0.805680
recall,0.203911,0.195501


**Observation:** While we don't see much overfitting here, the low `Recall` score indicates this model is failing to correctly identify Positive outcomes.

### Random Forest Classifier

In [42]:
rtc = RandomForestClassifier(random_state=42)

rtc.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [43]:
rtc_results = model_metric_results(rtc, X_train_scaled, y_train)
rtc_results

,Training_Score,Mean_CV_Score
roc_auc,0.686120,0.905740
recall,0.304469,0.709546


**Observation:** The fact that the cross-validation scores are significantly higher than the training scores indicates that while the model does better in predicting Positive outcomes on unseen data than it is on the training data. This could be due to a need to fine-tune parameters for of the model, or to resample the classes to help with the imbalance flagged earlier.

### Adaptive Boost Classifier

In [45]:
adaboost = AdaBoostClassifier(random_state=42)

adaboost.fit(X_train_scaled, y_train)

AdaBoostClassifier(random_state=42)

In [46]:
adaboost_results = model_metric_results(adaboost, X_train_scaled, y_train)
adaboost_results

,Training_Score,Mean_CV_Score
roc_auc,0.886231,0.853304
recall,0.379888,0.318310


**Observation:** While training and cross-validation scores are closer to each other, the poor Recall score indicates this model is not performing well in predicting Positive cases correctly.

### Rebalancing Target Classes

Since none of the models we've used have performed particularly well so far, and considering the moderate imbalance in the Target classes noted above (86% Negative, 14% Positive), we will try to rebalance those classes by synthetically oversampling the minority class (Negatives) using `SMOTE`. Then we will apply the vanilla models once again and see if this resampling affected the models' performances.

In [47]:
# Creating a function to store model results for easier comparison

models_results = {
    'models' : {},
    'results' : {}
}

def store_model_results(name, model, results, dict=models_results):
    dict['models'][name] = model
    dict['results'][name] = results

In [49]:
store_model_results('logreg', logreg, logreg_results)
store_model_results('rtc', rtc, rtc_results)
store_model_results('adaboost', adaboost, adaboost_results)

In [53]:
smote = SMOTE(random_state=42, sampling_strategy=0.3)

X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

In [55]:
# Checking the rebalanced classes

y_train_resampled.value_counts(normalize=True)

churn
0    0.769314
1    0.230686
Name: proportion, dtype: float64

#### Now we can run the vanilla models again on the resampled sets and see if it improved the models' performances

In [58]:
logreg_resampled = LogisticRegression(random_state=42)

logreg_resampled.fit(X_train_resampled, y_train_resampled)

logreg_resampled_results = model_metric_results(logreg_resampled, X_train_resampled, y_train_resampled)

In [59]:
rtc_resampled = RandomForestClassifier(random_state=42)

rtc_resampled.fit(X_train_resampled, y_train_resampled)

rtc_resampled_results = model_metric_results(rtc_resampled, X_train_resampled, y_train_resampled)

In [60]:
adaboost_resampled = AdaBoostClassifier(random_state=42)

adaboost_resampled.fit(X_train_resampled, y_train_resampled)

adaboost_resampled_results = model_metric_results(adaboost_resampled, X_train_resampled, y_train_resampled)

In [61]:
# Storing the results in our model_results dictionary for comparison

store_model_results('logreg_resampled', logreg_resampled, logreg_resampled_results)
store_model_results('rtc_resampled', rtc_resampled, rtc_resampled_results)
store_model_results('adaboost_resampled', adaboost_resampled, adaboost_resampled_results)

In [65]:
# Using for lop to display results for easier comparison

for model, results in models_results['results'].items():
    print(f' Results for {model} model')
    display(results)


 Results for logreg model


,Training_Score,Mean_CV_Score
roc_auc,0.818385,0.805680
recall,0.203911,0.195501


 Results for rtc model


,Training_Score,Mean_CV_Score
roc_auc,0.686120,0.905740
recall,0.304469,0.709546


 Results for adaboost model


...

,Training_Score,Mean_CV_Score
roc_auc,0.826100,0.819443
recall,0.376947,0.367599


 Results for rtc_resampled model


,Training_Score,Mean_CV_Score
roc_auc,1.0,0.962601
recall,1.0,0.814668


 Results for adaboost_resampled model


,Training_Score,Mean_CV_Score
roc_auc,0.897067,0.873263
recall,0.571651,0.468968


**Observation:** It seems over-sampling definitely increased the performance of all three of our models.

The `rtc_resampled` model performed best overall, with perfect ROC_AUC and Recall scores for the training sets and very high respective scores for the test sets.

We will print the results of both `rtc` and `rtc_resampled` below for convenience:

In [79]:
print('RTC Model Scores')
display(models_results['results']['rtc'])
print()
print('RTC_Resampled Model Scores')
display(models_results['results']['rtc_resampled'])

RTC Model Scores


,Training_Score,Mean_CV_Score
roc_auc,0.686120,0.905740
recall,0.304469,0.709546



RTC_Resampled Model Scores


,Training_Score,Mean_CV_Score
roc_auc,1.0,0.962601
recall,1.0,0.814668


**Note:** Though the perfect training scores may indicate overfitting in our rtc_resampled model, the high test scores indicate that this overfitting does not prevent the model from performing well on unseen data.

In [89]:
rtc_resampled_metrics = model_metric_results(rtc_resampled, X_test_scaled, y_test, 
                                           metrics=['roc_auc', 'recall', 'precision', 'f1', 'accuracy'], cv=False)

print('RTC Resampled Results')
display(rtc_resampled_metrics)

RTC Resampled Results


,Score
roc_auc,0.929427
recall,0.768000
precision,0.941176
f1,0.845815
accuracy,0.958034


In [91]:
print('RTC Resampled Results')
display(rtc_resampled_metrics.loc[['roc_auc', 'recall'], 'Score'])
print()
print('Training Data for Comparison')
display(rtc_resampled_results)

RTC Resampled Results


roc_auc    0.929427
recall     0.768000
Name: Score, dtype: float64


Training Data for Comparison


,Training_Score,Mean_CV_Score
roc_auc,1.0,0.962601
recall,1.0,0.814668


Though all the metrics displayed above are good, let's see if we can raise the Recall score by tuning the model's parameters.

### Tuning Hyperparameters

We will efficiently tune the hypermarameters of our chosen model using `GridSearchCV`.

In [96]:
# Looking at the lsit of parameters for the model
rtc.get_params()

{'bootstrap': True,
 'ccp_alpha': 0.0,
 'class_weight': None,
 'criterion': 'gini',
 'max_depth': None,
 'max_features': 'sqrt',
 'max_leaf_nodes': None,
 'max_samples': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'n_estimators': 100,
 'n_jobs': None,
 'oob_score': False,
 'random_state': 42,
 'verbose': 0,
 'warm_start': False}

In [ ]:
# Creating a dictionary containing variety of hyperparameters that GridSearchCV will run through
grid_params = {
    'n_estimators': [50, 100, 150, 200],
    'criterion': ['gini', 'entropy'],
    'max_features': [20, 30, 50],
    'min_samples_split': [2, 4, 8],
    'min_samples_leaf': [1, 3, 5]
}

rtc_grid_resample = GridSearchCV(
    estimator=rtc,
    param_grid=grid_params,
    cv=3,
    scoring='recall',
    n_jobs=-1
)

rtc_grid_resample.fit(X_train_resampled, y_train_resampled)

In [114]:
# Checking for the optimal combination of hyperparameters
rtc_grid_resample.best_params_

{'criterion': 'entropy', 'max_features': 20, 'n_estimators': 100}

In [115]:
rtc_tuned_resample_results = model_metric_results(rtc_grid_resample.best_estimator_, X_train_resampled, y_train_resampled)
rtc_tuned_resample_results

,Training_Score,Mean_CV_Score
roc_auc,1.0,0.959969
recall,1.0,0.822469


In [110]:
# Displaying training data for comparison
rtc_resampled_results

,Training_Score,Mean_CV_Score
roc_auc,1.0,0.962601
recall,1.0,0.814668


While the ROC-AUC score is pretty much identical at 96%, the Recall score is marginally better, having risen to 82% (which is 1% difference).

We will use this tuned model as our final model.

In [111]:
final_model = rtc_grid_resample.best_estimator_

Since this `final_model` has already been fitted on our training data, we can directly apply it to the  test data.

In [112]:
final_model_results = model_metric_results(final_model, X_test_scaled, y_test, 
                                           metrics=['roc_auc', 'recall', 'precision', 'f1', 'accuracy'], cv=False)

print('Final Model Results')
display(final_model_results)

Final Model Results


,Score
roc_auc,0.924745
recall,0.784000
precision,0.907407
f1,0.841202
accuracy,0.955635


# Conclusion

## Limitations

## Recommendations

## Next Steps